# Integrating dHT with Vision Transformer (ViT) Models

This notebook demonstrates how to integrate Differentiable Hierarchical Tokenization with Vision Transformer models.

## Overview

Standard ViT uses fixed-size patches (typically 16x16). dHT replaces this with adaptive tokenization:

```
Standard ViT Pipeline:
Image → Fixed Patches → Linear Projection → Transformer → Output

dHT-ViT Pipeline:
Image → dHT Tokenizer → dHT Extractor → dHT Embedder → Transformer → Output
```

### Key Benefits:
- Fewer tokens for simple regions
- More tokens for detailed areas
- Better semantic alignment
- Improved efficiency

In [ ]:
import torch
import torch.nn as nn
from dht.nn.transformer import dHTEncoder, dHTClassifier
from dht.tok.tokenizer import dHTTokenizer
from dht.tok.extractor import dHTExtractor
from dht.tok.embedder import dHTEmbedder
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Build a Complete dHT-ViT Model

The `dHTEncoder` class provides a complete transformer with adaptive tokenization.

In [ ]:
# Create a small dHT-ViT model
model = dHTEncoder.build(
    capacity='S',      # Small model (S/M/B/L/H)
    patch_size=16,     # Base patch size
    channels=3,        # RGB
    compute_grad=True, # Include gradient features
    num_cls_tokens=1,  # Number of classification tokens
).to(device)

print(f"Model created with:")
print(f"  Embed dim: {model.embed_dim}")
print(f"  Heads: {model.heads}")
print(f"  Depth: {model.depth}")
print(f"  Patch size: {model.patch_size}")

## 2. Forward Pass Through the Model

In [ ]:
# Create sample input
batch_size = 2
img = torch.randn(batch_size, 3, 224, 224).to(device)

# Forward pass
with torch.no_grad():
    model.eval()
    result = model(img)

print(f"\nModel output:")
print(f"  Features shape: {result.fV.shape}")
print(f"  Segmentation shape: {result.seg.shape if result.seg is not None else 'None'}")
print(f"  Attention mask shape: {result.amask.shape if result.amask is not None else 'None'}")

## 3. Understanding the dHT Pipeline Components

Let's break down each component:

In [ ]:
# Step 1: Tokenization
tokenizer_result = model.tokenizer(img)
print(f"1. Tokenizer output:")
print(f"   Tokens: {tokenizer_result.nV}")
print(f"   Features: {tokenizer_result.fV.shape}")

# Step 2: Feature Extraction
extractor_result = model.extractor(tokenizer_result)
print(f"\n2. Extractor output:")
print(f"   Features: {extractor_result.fV.shape}")

# Step 3: Embedding
embedder_result = model.embedder(extractor_result)
print(f"\n3. Embedder output:")
print(f"   Embedded features: {embedder_result.fV.shape}")
print(f"   Ready for transformer!")

## 4. Classification with dHT

For image classification, use the `dHTClassifier` which adds a classification head.

In [ ]:
# Create classifier
classifier = dHTClassifier(
    embed_dim=384,
    patch_size=16,
    heads=6,
    depth=12,
    n_classes=1000,  # ImageNet classes
    channels=3,
    compute_grad=True,
).to(device)

# Forward pass
with torch.no_grad():
    classifier.eval()
    logits = classifier(img)

print(f"Classification logits shape: {logits.shape}")
print(f"Batch size: {logits.shape[0]}, Classes: {logits.shape[1]}")

## 5. Adaptive Token Count

One key feature of dHT is adaptive token count based on image complexity.

In [ ]:
# Create images with different complexity
simple_img = torch.ones(1, 3, 224, 224).to(device)  # Uniform
complex_img = torch.randn(1, 3, 224, 224).to(device)  # Random noise

with torch.no_grad():
    model.eval()
    simple_result = model.tokenizer(simple_img)
    complex_result = model.tokenizer(complex_img)

print(f"Token count comparison:")
print(f"  Simple image: {simple_result.nV} tokens")
print(f"  Complex image: {complex_result.nV} tokens")
print(f"\nFixed ViT would use: {(224//16)**2} = 196 tokens for both")

## 6. Controlling Token Count

You can control the target number of tokens:

In [ ]:
# Try different target token counts
targets = [50, 100, 200]

for target in targets:
    with torch.no_grad():
        result = model(img, target=target)
    actual_tokens = result.amask.sum(dim=1).float().mean().item()
    print(f"Target: {target} tokens, Actual: {actual_tokens:.1f} tokens (avg)")

## 7. Attention Mask

dHT produces variable-length token sequences. The attention mask handles this:

In [ ]:
with torch.no_grad():
    result = model(img)

print(f"Attention mask shape: {result.amask.shape}")
print(f"\nFor each sample in batch:")
for i in range(batch_size):
    n_tokens = result.amask[i].sum().item()
    print(f"  Sample {i}: {n_tokens} active tokens")

## 8. Model Sizes

dHT supports different model capacities:

In [ ]:
capacities = ['S', 'M', 'B', 'L']

for cap in capacities:
    m = dHTEncoder.build(capacity=cap, patch_size=16)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{cap}: {n_params/1e6:.1f}M parameters, "
          f"dim={m.embed_dim}, heads={m.heads}, depth={m.depth}")

## Summary

This notebook covered:
1. Building dHT-ViT models
2. Understanding the tokenization pipeline
3. Classification with dHT
4. Adaptive token counts
5. Attention masking for variable-length sequences
6. Different model capacities

### Next: Training Pipeline (Notebook 03)